<a href="https://colab.research.google.com/github/arshad831/zain_2026/blob/main/Zain_Class8_RAG_and_LangSmith_Evaluation_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Zain Jordan Class 4 + Class 4.5

# RAG Assistant + LangSmith Evaluation

This Colab notebook combines:

1. **Class 4: RAG Assistant from Telecom Database Content**
2. **Class 4.5: LangSmith Evaluation for RAG**

## What this notebook does

```text
SQLite Customer 360 Database
        ↓
Selected rows become documents
        ↓
OpenAI embeddings
        ↓
InMemoryVectorStore
        ↓
Retriever
        ↓
RAG answer
        ↓
LangSmith golden dataset
        ↓
LangSmith evaluation experiment
```

## Assumptions

Before running this notebook:

1. Upload the database to Colab:

```text
/content/zain_customer_360_ai_demo.db
```

2. Save these keys in Google Colab Secrets:

```text
OPENAI_API_KEY
LANGSMITH_API_KEY
```

Optional:

```text
LANGSMITH_PROJECT
```

If `LANGSMITH_PROJECT` is not set, the notebook will use:

```text
zain-class4-rag-langsmith-eval
```


In [1]:
# ============================================================
# 1. Install packages
# ============================================================

!pip install -q -U \
    langchain \
    langchain-openai \
    langchain-community \
    langsmith \
    pandas \
    sqlalchemy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.6/399.6 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.

In [11]:
try:
    from google.colab import userdata
    openai_key = userdata.get("openai")
    if openai_key:
        os.environ["OPENAI_API_KEY"] = openai_key
        print("OpenAI API key loaded from Colab Secrets.")
    else:
        print("OPENAI_API_KEY not found in Colab Secrets.")
except Exception:
    print("Not running in Google Colab, or Colab Secrets not available.")

if not os.environ.get("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY is not set. Agent cells will not run until it is configured.")
else:
    print("OPENAI_API_KEY is available.")


OpenAI API key loaded from Colab Secrets.
OPENAI_API_KEY is available.


In [3]:
# ============================================================
# 2. Load API keys from Colab Secrets and set paths
# ============================================================

import os
from pathlib import Path

DB_PATH = "/content/zain_customer_360_ai_demo.db"
LANGSMITH_PROJECT = "zain-class4-rag-langsmith-eval"

try:
    from google.colab import userdata

    openai_key = userdata.get("openai")
    langsmith_key = userdata.get("LANGSMITH_API_KEY")
    langsmith_project = userdata.get("LANGSMITH_PROJECT")

    if openai_key:
        os.environ["OPENAI_API_KEY"] = openai_key
        print("OPENAI_API_KEY loaded from Colab Secrets.")
    else:
        print("OPENAI_API_KEY not found in Colab Secrets.")

    if langsmith_key:
        os.environ["LANGSMITH_API_KEY"] = langsmith_key
        print("LANGSMITH_API_KEY loaded from Colab Secrets.")
    else:
        print("LANGSMITH_API_KEY not found in Colab Secrets.")

    if langsmith_project:
        LANGSMITH_PROJECT = langsmith_project

except Exception:
    print("Not running in Colab, or Colab Secrets not available.")

# LangSmith tracing setup
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT

print("LangSmith project:", os.environ["LANGSMITH_PROJECT"])

if Path(DB_PATH).exists():
    print("Database found:", DB_PATH)
else:
    print("Database not found. Please upload zain_customer_360_ai_demo.db to /content/")


Not running in Colab, or Colab Secrets not available.
LangSmith project: zain-class4-rag-langsmith-eval
Database found: /content/zain_customer_360_ai_demo.db


In [4]:
# ============================================================
# 3. Inspect database tables
# ============================================================

import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)

tables_df = pd.read_sql_query(
    '''
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    ''',
    conn
)

display(tables_df)

conn.close()


,name
0,accounts
1,addons
2,call_detail_records
3,campaigns
4,complaints
5,customer_campaign_responses
6,customer_churn_scores
7,customer_monthly_summary
8,customer_satisfaction
9,customer_value_segments


In [5]:
# ============================================================
# 4. Convert selected telecom database rows into RAG documents
# ============================================================
# Class 4 teaching point:
# Not every database table should become RAG content.
# RAG is useful for meaning-rich content such as plans, add-ons,
# campaigns, complaints, support notes, and satisfaction feedback.

import sqlite3
import pandas as pd
from langchain_core.documents import Document

RAG_TABLES = [
    "plans",
    "addons",
    "campaigns",
    "complaints",
    "support_interactions",
    "customer_satisfaction",
]

MAX_ROWS_PER_TABLE = {
    "plans": 500,
    "addons": 500,
    "campaigns": 500,
    "complaints": 300,
    "support_interactions": 300,
    "customer_satisfaction": 300,
}


def get_existing_tables(db_path):
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query(
        '''
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name;
        ''',
        conn
    )
    conn.close()
    return set(df["name"].tolist())


def row_to_document_text(table_name, row_dict):
    lines = [f"Source Table: {table_name}"]

    for key, value in row_dict.items():
        if value is not None and str(value).strip() != "":
            pretty_key = key.replace("_", " ").title()
            lines.append(f"{pretty_key}: {value}")

    return "\n".join(lines)


def build_documents_from_database(db_path):
    existing_tables = get_existing_tables(db_path)
    documents = []

    conn = sqlite3.connect(db_path)

    for table_name in RAG_TABLES:
        if table_name not in existing_tables:
            print(f"Skipping missing table: {table_name}")
            continue

        limit = MAX_ROWS_PER_TABLE.get(table_name, 200)
        query = f"SELECT * FROM {table_name} LIMIT {limit};"
        df = pd.read_sql_query(query, conn)

        for idx, row in df.iterrows():
            row_dict = row.to_dict()
            text = row_to_document_text(table_name, row_dict)

            metadata = {
                "source": "zain_customer_360_ai_demo.db",
                "table": table_name,
                "row_index": int(idx),
            }

            # Add possible ID columns to metadata for traceability
            for col in df.columns:
                if col.endswith("_id") and col in row_dict:
                    metadata[col] = str(row_dict[col])

            documents.append(Document(page_content=text, metadata=metadata))

        print(f"Added {len(df)} documents from table: {table_name}")

    conn.close()
    return documents


documents = build_documents_from_database(DB_PATH)

print("\nTotal RAG documents:", len(documents))
print("\nSample document:")
print(documents[0].page_content[:1000] if documents else "No documents created.")
print("\nSample metadata:")
print(documents[0].metadata if documents else {})


Added 25 documents from table: plans
Added 20 documents from table: addons
Added 20 documents from table: campaigns
Added 300 documents from table: complaints
Added 300 documents from table: support_interactions
Added 300 documents from table: customer_satisfaction

Total RAG documents: 965

Sample document:
Source Table: plans
Plan Id: 1
Plan Name: Go 15 Plus
Plan Category: Mobile Postpaid
Service Type: Mobile Voice
Monthly Fee Jod: 15.0
Data Allowance Gb: 20.0
Local Minutes: 1000
International Minutes: 30
Roaming Minutes: 0
Sms Allowance: 100
Technology: 4G
Contract Months: 12
Data Carryover Flag: 1
Is Business Plan: 0
Status: Active

Sample metadata:
{'source': 'zain_customer_360_ai_demo.db', 'table': 'plans', 'row_index': 0, 'plan_id': '1'}


In [12]:
# ============================================================
# 5. Create embeddings, vector store, and retriever
# ============================================================

from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = InMemoryVectorStore.from_documents(
    documents=documents,
    embedding=embeddings,
)

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print("Vector store and retriever are ready.")


Vector store and retriever are ready.


In [13]:
# ============================================================
# 6. Test retrieval before generation
# ============================================================
# Class 4 teaching point:
# If retrieval is poor, the final RAG answer will also be poor.

test_query = "best offer for a heavy data user with roaming needs"

retrieved_docs = retriever.invoke(test_query)

print("Retrieved documents:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, start=1):
    print("\n" + "=" * 80)
    print("DOC", i)
    print("Metadata:", doc.metadata)
    print(doc.page_content[:1000])


/usr/local/lib/python3.12/dist-packages/langsmith/client.py:640: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


Retrieved documents: 4

DOC 1
Metadata: {'source': 'zain_customer_360_ai_demo.db', 'table': 'plans', 'row_index': 5, 'plan_id': '6'}
Source Table: plans
Plan Id: 6
Plan Name: Family Max
Plan Category: Mobile Postpaid
Service Type: Mobile Voice
Monthly Fee Jod: 55.0
Data Allowance Gb: 200.0
Local Minutes: 15000
International Minutes: 100
Roaming Minutes: 50
Sms Allowance: 2000
Technology: 5G
Contract Months: 24
Data Carryover Flag: 1
Is Business Plan: 0
Status: Active

DOC 2
Metadata: {'source': 'zain_customer_360_ai_demo.db', 'table': 'plans', 'row_index': 17, 'plan_id': '18'}
Source Table: plans
Plan Id: 18
Plan Name: MiFi Student Pack
Plan Category: 4G Broadband
Service Type: Mobile Data
Monthly Fee Jod: 14.0
Data Allowance Gb: 120.0
Local Minutes: 0
International Minutes: 0
Roaming Minutes: 0
Sms Allowance: 0
Technology: 4G
Contract Months: 0
Data Carryover Flag: 0
Is Business Plan: 0
Status: Active

DOC 3
Metadata: {'source': 'zain_customer_360_ai_demo.db', 'table': 'plans', 'row_i

In [14]:
# ============================================================
# 7. Create the RAG answer function
# ============================================================

from langchain_openai import ChatOpenAI
from langsmith import traceable

RAG_MODEL_NAME = "gpt-4.1-mini"

rag_llm = ChatOpenAI(model=RAG_MODEL_NAME)


def format_docs(docs):
    formatted = []

    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("table", "unknown")
        formatted.append(
            f"[Document {i} | Source table: {source}]\n{doc.page_content}"
        )

    return "\n\n".join(formatted)


@traceable(name="zain_class4_rag_assistant")
def rag_answer(question: str) -> dict:
    docs = retriever.invoke(question)
    context = format_docs(docs)

    system_prompt = f'''
You are a Zain Jordan telecom RAG assistant.

Use only the retrieved context to answer the user question.

Rules:
- Do not invent plan names, prices, campaigns, add-ons, policies, or compensation.
- If the retrieved context is not enough, say what is missing.
- Mention the evidence/source tables used.
- Give a practical business recommendation.
- Keep the answer clear and structured.

Retrieved context:
{context}
'''.strip()

    response = rag_llm.invoke([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ])

    return {
        "answer": response.content,
        "documents": docs,
        "source_tables": sorted(list({doc.metadata.get("table", "unknown") for doc in docs})),
    }


sample_question = "Which is the best plan for heavy data users who are likely to churn?"

sample_result = rag_answer(sample_question)

print(sample_result["answer"])
print("\nSource tables:", sample_result["source_tables"])


For heavy data users who are at risk of churning, the best plan recommendation would be the "VIP Unlimited" plan based on the retrieved context. Here's why:

1. High Data Allowance: VIP Unlimited offers 300 GB of data per month, which is suitable for heavy data users.
2. Additional Benefits: The plan includes 20,000 local minutes, 250 international minutes, 100 roaming minutes, and 2,000 SMS, providing comprehensive value beyond data.
3. 5G Technology: It provides the latest 5G technology ensuring high-speed internet access.
4. Data Carryover: This plan allows unused data to be carried over, offering more flexibility and perceived value.
5. Contract: It has a 24-month contract, so retention strategies may be needed for those likely to churn.

Business Recommendation:
- Highlight the high data volume and added value (minutes, SMS, and international use) to customers identified as heavy data users.
- Implement loyalty or retention offers on top of this plan for at-risk customers, such as

In [15]:
# ============================================================
# 8. Create 30-question golden dataset in JSON format
# ============================================================

import json
import pandas as pd

golden_examples = [{'id': 'N1', 'level': 'normal', 'question': 'Recommend a plan for a heavy mobile data user.', 'reference_answer': 'Recommend a high-data mobile plan. The answer should mention data allowance, monthly fee if available, and explain why it fits heavy usage.', 'expected_sources': ['plans'], 'skill_tested': 'basic plan recommendation'}, {'id': 'N2', 'level': 'normal', 'question': 'Which add-on is suitable for a customer who travels frequently?', 'reference_answer': 'Recommend a roaming-related add-on if available. The answer should explain the travel or roaming fit and mention limitations if no roaming add-on is found.', 'expected_sources': ['addons'], 'skill_tested': 'add-on retrieval'}, {'id': 'N3', 'level': 'normal', 'question': 'What plan is suitable for a family with high data usage?', 'reference_answer': 'Recommend a family or high-data plan. The answer should mention why it fits family or shared high-data usage.', 'expected_sources': ['plans'], 'skill_tested': 'family plan retrieval'}, {'id': 'N4', 'level': 'normal', 'question': 'Recommend a campaign for a high-value customer.', 'reference_answer': 'Retrieve campaigns targeting high-value, VIP, premium, or retention customers. The answer should not recommend a generic campaign if a more relevant high-value campaign exists.', 'expected_sources': ['campaigns'], 'skill_tested': 'campaign retrieval'}, {'id': 'N5', 'level': 'normal', 'question': 'Find support guidance for a customer confused about billing.', 'reference_answer': 'Retrieve billing-related complaint or support records and summarize a helpful support response. The answer should be grounded in retrieved billing support context.', 'expected_sources': ['complaints', 'support_interactions'], 'skill_tested': 'support guidance'}, {'id': 'N6', 'level': 'normal', 'question': 'Which add-on is best for extra data usage?', 'reference_answer': 'Recommend an extra-data add-on with data amount and fee if available. The answer should explain why it fits extra data usage.', 'expected_sources': ['addons'], 'skill_tested': 'data add-on recommendation'}, {'id': 'N7', 'level': 'normal', 'question': 'Recommend a plan for a low-cost customer.', 'reference_answer': 'Recommend a lower-cost or value-oriented plan if present. The answer should avoid premium recommendations unless no cheaper option is found.', 'expected_sources': ['plans'], 'skill_tested': 'cost-sensitive plan recommendation'}, {'id': 'N8', 'level': 'normal', 'question': 'What campaign should be suggested to prepaid customers?', 'reference_answer': 'Retrieve prepaid-targeted campaigns if present. If the retrieved context does not mention prepaid, the answer should say the data is insufficient.', 'expected_sources': ['campaigns'], 'skill_tested': 'segment-specific campaign retrieval'}, {'id': 'N9', 'level': 'normal', 'question': 'Summarize common complaint themes related to network issues.', 'reference_answer': 'Summarize network-related complaints or support notes. The answer should focus on retrieved network issue context and avoid unsupported claims.', 'expected_sources': ['complaints', 'support_interactions'], 'skill_tested': 'complaint theme summarization'}, {'id': 'N10', 'level': 'normal', 'question': 'Recommend a customer-care message for a billing complaint.', 'reference_answer': 'Draft a short, professional, customer-friendly message grounded in billing complaint or support context. The answer should not promise compensation unless supported by retrieved context.', 'expected_sources': ['complaints', 'support_interactions'], 'skill_tested': 'grounded message generation'}, {'id': 'I1', 'level': 'intermediate', 'question': 'Which is the best plan for heavy data users who are likely to churn?', 'reference_answer': 'Recommend a high-data plan or relevant retention offer. The answer must explain why it fits heavy data usage and churn risk, and should cite plan or campaign evidence.', 'expected_sources': ['plans', 'campaigns'], 'skill_tested': 'RAG recommendation plus retention reasoning'}, {'id': 'I2', 'level': 'intermediate', 'question': 'Recommend an offer for a VIP customer with roaming usage.', 'reference_answer': 'Retrieve roaming add-ons and high-value or VIP campaigns if available. The answer should recommend a combined plan, add-on, or campaign and explain the fit.', 'expected_sources': ['addons', 'campaigns'], 'skill_tested': 'combined add-on and campaign recommendation'}, {'id': 'I3', 'level': 'intermediate', 'question': 'A customer complains about roaming charges. What plan or add-on should we suggest?', 'reference_answer': 'Recommend a roaming add-on or relevant plan if available. The answer should also include support guidance for billing or roaming concerns if retrieved.', 'expected_sources': ['addons', 'complaints', 'support_interactions'], 'skill_tested': 'complaint-aware recommendation'}, {'id': 'I4', 'level': 'intermediate', 'question': 'Which campaign fits a customer with high churn risk and high value?', 'reference_answer': 'Recommend a retention or high-value campaign. The answer should not suggest a generic low-value campaign if a more relevant retention campaign is available.', 'expected_sources': ['campaigns'], 'skill_tested': 'retention campaign recommendation'}, {'id': 'I5', 'level': 'intermediate', 'question': 'Recommend a plan for a customer who uses video streaming heavily.', 'reference_answer': 'Retrieve a high-data, 5G, or large-allowance plan. The answer should explain why the plan supports streaming-heavy behavior.', 'expected_sources': ['plans'], 'skill_tested': 'usage-based plan recommendation'}, {'id': 'I6', 'level': 'intermediate', 'question': 'What support response should we give for repeated billing confusion?', 'reference_answer': 'Summarize billing confusion patterns from support or complaint records and draft a professional support response. The answer should avoid unsupported compensation promises.', 'expected_sources': ['complaints', 'support_interactions'], 'skill_tested': 'support response and grounding'}, {'id': 'I7', 'level': 'intermediate', 'question': 'Recommend a plan and add-on combination for a frequent traveler.', 'reference_answer': 'Recommend a suitable base plan plus roaming or travel add-on if available. The answer should explain why both are needed.', 'expected_sources': ['plans', 'addons'], 'skill_tested': 'multi-source recommendation'}, {'id': 'I8', 'level': 'intermediate', 'question': 'Which campaign should target customers unhappy with service quality?', 'reference_answer': 'Retrieve campaigns or offers related to retention, satisfaction, service recovery, or experience improvement if available. The answer should mention uncertainty if the retrieved context is weak.', 'expected_sources': ['campaigns', 'customer_satisfaction', 'support_interactions'], 'skill_tested': 'experience-based campaign matching'}, {'id': 'I9', 'level': 'intermediate', 'question': 'Suggest a retention action for a customer with poor satisfaction feedback.', 'reference_answer': 'Use satisfaction, support, and campaign context to suggest a retention action. The answer should be grounded and should not invent customer-specific facts.', 'expected_sources': ['customer_satisfaction', 'campaigns', 'support_interactions'], 'skill_tested': 'retention action recommendation'}, {'id': 'I10', 'level': 'intermediate', 'question': 'Find the best recommendation for a customer who wants more data but lower monthly cost.', 'reference_answer': 'Balance data allowance and price. The answer should discuss trade-offs and recommend the closest matching plan or add-on based on retrieved evidence.', 'expected_sources': ['plans', 'addons'], 'skill_tested': 'trade-off reasoning'}, {'id': 'A1', 'level': 'advanced_edge_case', 'question': 'Recommend the cheapest unlimited global roaming plan.', 'reference_answer': 'If no unlimited global roaming plan is present in the retrieved context, the answer should clearly say there is not enough evidence and avoid inventing one.', 'expected_sources': ['plans', 'addons'], 'skill_tested': 'hallucination control'}, {'id': 'A2', 'level': 'advanced_edge_case', 'question': 'Which plan guarantees 100% network coverage in Jordan?', 'reference_answer': 'The answer should not claim any plan guarantees 100% coverage unless the retrieved context explicitly states this. It should explain that coverage guarantees require network data or policy evidence.', 'expected_sources': ['plans', 'support_interactions'], 'skill_tested': 'unsupported claim detection'}, {'id': 'A3', 'level': 'advanced_edge_case', 'question': 'Give a free premium upgrade to a high churn customer.', 'reference_answer': 'The answer should not promise a free premium upgrade unless a retrieved campaign or policy supports it. It may suggest escalation or eligibility check.', 'expected_sources': ['campaigns'], 'skill_tested': 'policy-safe response'}, {'id': 'A4', 'level': 'advanced_edge_case', 'question': 'Show all customers who need this offer.', 'reference_answer': 'The answer should identify this as a SQL or segmentation task, not a pure RAG task. It should explain that customer selection needs structured database filtering.', 'expected_sources': ['campaigns'], 'skill_tested': 'architecture boundary'}, {'id': 'A5', 'level': 'advanced_edge_case', 'question': 'Which campaign had the highest conversion rate?', 'reference_answer': 'The answer should identify this as SQL analytics, not RAG-only. It should recommend using SQL Agent over campaign response tables.', 'expected_sources': ['campaigns', 'customer_campaign_responses'], 'skill_tested': 'SQL vs RAG routing'}, {'id': 'A6', 'level': 'advanced_edge_case', 'question': 'Recommend a plan for a customer without mentioning evidence.', 'reference_answer': 'The assistant should still include evidence or source context because the system is designed to produce grounded RAG answers.', 'expected_sources': ['plans', 'addons'], 'skill_tested': 'source visibility'}, {'id': 'A7', 'level': 'advanced_edge_case', 'question': 'A customer wants unlimited data, lowest price, and premium roaming. What should we recommend?', 'reference_answer': 'The answer should explain trade-offs. If no exact match exists, it should suggest the closest option and clearly state limitations.', 'expected_sources': ['plans', 'addons'], 'skill_tested': 'trade-off and limitation handling'}, {'id': 'A8', 'level': 'advanced_edge_case', 'question': 'Create a compensation promise for network outage dissatisfaction.', 'reference_answer': 'The answer should avoid promising compensation unless retrieved support or complaint context explicitly allows it. It should suggest escalation or investigation.', 'expected_sources': ['support_interactions', 'complaints'], 'skill_tested': 'safe customer communication'}, {'id': 'A9', 'level': 'advanced_edge_case', 'question': 'Recommend an enterprise business plan for a residential prepaid customer.', 'reference_answer': 'The answer should identify the mismatch and recommend based on the correct customer segment if data supports it.', 'expected_sources': ['plans', 'campaigns'], 'skill_tested': 'segment consistency'}, {'id': 'A10', 'level': 'advanced_edge_case', 'question': 'Summarize all plan prices and rank them cheapest to most expensive.', 'reference_answer': 'The answer should identify this as a structured SQL/table ranking task, not a semantic RAG task. It should recommend using SQL Agent.', 'expected_sources': ['plans'], 'skill_tested': 'RAG vs SQL boundary'}]

json_path = "/content/zain_rag_langsmith_golden_dataset_30.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(golden_examples, f, indent=2, ensure_ascii=False)

golden_df = pd.DataFrame(golden_examples)

display(golden_df)

print("Golden dataset saved to:", json_path)
print("Total examples:", len(golden_examples))


,id,level,question,reference_answer,expected_sources,skill_tested
0,N1,normal,Recommend a plan for a heavy mobile data user.,Recommend a high-data mobile plan. The answer ...,[plans],basic plan recommendation
1,N2,normal,Which add-on is suitable for a customer who tr...,Recommend a roaming-related add-on if availabl...,[addons],add-on retrieval
2,N3,normal,What plan is suitable for a family with high d...,Recommend a family or high-data plan. The answ...,[plans],family plan retrieval
3,N4,normal,Recommend a campaign for a high-value customer.,"Retrieve campaigns targeting high-value, VIP, ...",[campaigns],campaign retrieval
4,N5,normal,Find support guidance for a customer confused ...,Retrieve billing-related complaint or support ...,"[complaints, support_interactions]",support guidance
5,N6,normal,Which add-on is best for extra data usage?,Recommend an extra-data add-on with data amoun...,[addons],data add-on recommendation
6,N7,normal,Recommend a plan for a low-cost customer.,Recommend a lower-cost or value-oriented plan ...,[plans],cost-sensitive plan recommendation
7,N8,normal,What campaign should be suggested to prepaid c...,Retrieve prepaid-targeted campaigns if present...,[campaigns],segment-specific campaign retrieval
8,N9,normal,Summarize common complaint themes related to n...,Summarize network-related complaints or suppor...,"[complaints, support_interactions]",complaint theme summarization
9,N10,normal,Recommend a customer-care message for a billin...,"Draft a short, professional, customer-friendly...","[complaints, support_interactions]",grounded message generation


Golden dataset saved to: /content/zain_rag_langsmith_golden_dataset_30.json
Total examples: 30


In [17]:
import os

try:
    from google.colab import userdata

    raw_key = userdata.get("LANGSMITH_API_KEY")

    print("Secret found:", raw_key is not None)

    if raw_key:
        print("Key length:", len(raw_key))
        print("Key starts with:", raw_key[:6])
        print("Key ends with:", raw_key[-4:])
        print("Has leading/trailing spaces:", raw_key != raw_key.strip())

        clean_key = raw_key.strip().strip('"').strip("'")

        os.environ["LANGSMITH_API_KEY"] = clean_key
        os.environ["LANGSMITH_TRACING"] = "true"
        os.environ["LANGSMITH_PROJECT"] = "zain-class-4-rag-evaluation"

        # Clear wrong endpoint if previously set
        os.environ.pop("LANGSMITH_ENDPOINT", None)

        print("LANGSMITH_API_KEY loaded into environment:", bool(os.environ.get("LANGSMITH_API_KEY")))

except Exception as e:
    print("Could not read Colab secret.")
    print(type(e).__name__, e)

Secret found: True
Key length: 51
Key starts with: lsv2_p
Key ends with: 6da3
Has leading/trailing spaces: False
LANGSMITH_API_KEY loaded into environment: True


In [18]:
from langsmith import Client

client = Client()

try:
    # This forces a real API call
    data = list(client.list_datasets(limit=1))
    print("✅ LangSmith authentication successful.")
except Exception as e:
    print("❌ LangSmith authentication failed.")
    print(type(e).__name__)
    print(e)

✅ LangSmith authentication successful.


In [19]:
# ============================================================
# 9. Create or reuse LangSmith dataset
# ============================================================
# This uploads the 30 golden examples to LangSmith.
# Each example has:
# - inputs: question
# - outputs: reference answer
# - metadata: level, expected sources, skill tested

from langsmith import Client
from datetime import datetime

client = Client()

DATASET_NAME = "Zain Class 4 RAG Plan Recommendation Golden Dataset - 30Q"


def get_or_create_dataset(dataset_name):
    try:
        dataset = client.create_dataset(
            dataset_name=dataset_name,
            description=(
                "Golden dataset for Zain Jordan Class 4 RAG assistant evaluation. "
                "Contains normal, intermediate, and advanced edge-case questions."
            ),
        )
        print("Created new LangSmith dataset:", dataset.name)
        return dataset

    except Exception as e:
        print("Dataset may already exist. Trying to reuse it.")
        try:
            return client.read_dataset(dataset_name=dataset_name)
        except Exception:
            raise e


dataset = get_or_create_dataset(DATASET_NAME)

# Prepare LangSmith examples
ls_examples = []

for ex in golden_examples:
    ls_examples.append({
        "inputs": {
            "question": ex["question"]
        },
        "outputs": {
            "answer": ex["reference_answer"]
        },
        "metadata": {
            "id": ex["id"],
            "level": ex["level"],
            "expected_sources": ex["expected_sources"],
            "skill_tested": ex["skill_tested"],
        }
    })

# Add examples. If they already exist, this may create duplicates.
# For live teaching, this is okay once. For repeated runs, create a new dataset name or clean old dataset.
try:
    client.create_examples(
        dataset_id=dataset.id,
        examples=ls_examples
    )
    print("Uploaded examples to LangSmith:", len(ls_examples))
except Exception as e:
    print("Could not upload examples. This may happen if examples already exist.")
    print("Error:", e)

print("Dataset name:", DATASET_NAME)


Created new LangSmith dataset: Zain Class 4 RAG Plan Recommendation Golden Dataset - 30Q
Uploaded examples to LangSmith: 30
Dataset name: Zain Class 4 RAG Plan Recommendation Golden Dataset - 30Q


In [20]:
# ============================================================
# 10. Define LangSmith evaluation target
# ============================================================
# LangSmith will call this function for each dataset example.

def target(inputs: dict) -> dict:
    result = rag_answer(inputs["question"])

    # LangSmith evaluators will look at:
    # - answer
    # - documents
    # - source_tables
    return {
        "answer": result["answer"],
        "documents": result["documents"],
        "source_tables": result["source_tables"],
    }


In [21]:
# ============================================================
# 11. Define simple RAG evaluators
# ============================================================
# These use an LLM as judge.
# They check:
# 1. correctness vs reference answer
# 2. groundedness vs retrieved documents
# 3. retrieval/source relevance

from typing_extensions import TypedDict, Annotated
from langchain_openai import ChatOpenAI

grader_llm = ChatOpenAI(model="gpt-4.1-mini")


class BooleanGrade(TypedDict):
    explanation: Annotated[str, ..., "Brief explanation for the grade"]
    score: Annotated[bool, ..., "True if it passes, False otherwise"]


structured_grader = grader_llm.with_structured_output(BooleanGrade)


def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    prompt = f'''
You are evaluating a RAG assistant.

Grade whether the generated answer satisfies the reference answer criteria.

Question:
{inputs["question"]}

Reference answer criteria:
{reference_outputs["answer"]}

Generated answer:
{outputs["answer"]}

Return True if the generated answer mostly satisfies the reference answer criteria.
Return False if it misses the main requirement or contradicts it.
'''.strip()

    grade = structured_grader.invoke(prompt)
    return grade["score"]


def groundedness_evaluator(inputs: dict, outputs: dict) -> bool:
    docs = outputs.get("documents", [])
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f'''
You are evaluating groundedness.

The generated answer should be supported by the retrieved context.
It should not invent unsupported plan names, add-ons, campaigns, prices, policies, or compensation.

Question:
{inputs["question"]}

Retrieved context:
{context}

Generated answer:
{outputs["answer"]}

Return True if the answer is mostly grounded in the retrieved context.
Return False if it makes important unsupported claims.
'''.strip()

    grade = structured_grader.invoke(prompt)
    return grade["score"]


def retrieval_relevance_evaluator(inputs: dict, outputs: dict) -> bool:
    docs = outputs.get("documents", [])
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f'''
You are evaluating retrieval relevance.

Question:
{inputs["question"]}

Retrieved documents:
{context}

Return True if the retrieved documents are relevant to the question.
Return False if the retrieved documents are mostly unrelated.
'''.strip()

    grade = structured_grader.invoke(prompt)
    return grade["score"]


In [22]:
# ============================================================
# 12. Run a quick local test before LangSmith evaluation
# ============================================================

local_test = target({"question": "Which is the best plan for heavy data users who are likely to churn?"})

print(local_test["answer"])
print("\nSource tables:", local_test["source_tables"])


For heavy data users likely to churn, the best plan should offer a high data allowance, attractive benefits, and some flexibility to retain the customer.

Based on the available plans:

1. VIP Unlimited (Plan Id 5)
   - Data Allowance: 300 GB
   - Monthly Fee: 75 JOD
   - Extras: 20,000 local minutes, 250 international minutes, 100 roaming minutes, 2000 SMS
   - Technology: 5G
   - Contract: 24 months
   - Data Carryover: Yes
   - Mobile Postpaid plan

2. Family Max (Plan Id 6)
   - Data Allowance: 200 GB
   - Monthly Fee: 55 JOD
   - Extras: 15,000 local minutes, 100 international minutes, 50 roaming minutes, 2000 SMS
   - Technology: 5G
   - Contract: 24 months
   - Data Carryover: Yes
   - Mobile Postpaid plan

3. 4G Broadband 500GB (Plan Id 17)
   - Data Allowance: 500 GB
   - Monthly Fee: 19 JOD
   - Home Internet service
   - Technology: 4G
   - Contract: 12 months
   - No data carryover

4. Prepaid 5G Lite (Plan Id 10)
   - Data Allowance: 35 GB
   - Monthly Fee: 12 JOD
   - Mob

In [25]:
RUN_FULL_DATASET = False
QUICK_RUN_COUNT = 30

if RUN_FULL_DATASET:
    data_for_eval = DATASET_NAME
    experiment_prefix = "zain-rag-full-30q"
else:
    # Fetch actual LangSmith Example objects from the dataset
    all_examples_from_langsmith = list(client.list_examples(dataset_id=dataset.id))
    data_for_eval = all_examples_from_langsmith[:QUICK_RUN_COUNT]
    experiment_prefix = "zain-rag-quick-demo-8q"


experiment_results = client.evaluate(
    target,
    data=data_for_eval,
    evaluators=[
        correctness_evaluator,
        groundedness_evaluator,
        retrieval_relevance_evaluator,
    ],
    experiment_prefix=experiment_prefix,
    max_concurrency=2,
    metadata={
        "rag_model": RAG_MODEL_NAME,
        "embedding_model": "text-embedding-3-small",
        "vector_store": "InMemoryVectorStore",
        "retriever_k": 4,
        "class_module": "Class 4 + Class 4.5",
    },
)

print(experiment_results)

try:
    results_df = experiment_results.to_pandas()
    display(results_df)
except Exception as e:
    print("Could not convert experiment results to pandas.")
    print(e)

View the evaluation results for experiment: 'zain-rag-quick-demo-8q-4732252d' at:
https://smith.langchain.com/o/c8f8810e-4941-552b-aef3-15ad938ead98/datasets/5289d840-3c22-4047-a686-f0a7e20eed18/compare?selectedSessions=79020a92-fd51-48d9-8715-e8e722cb53ec




0it [00:00, ?it/s]

<ExperimentResults zain-rag-quick-demo-8q-4732252d>


,inputs.question,outputs.answer,outputs.documents,outputs.source_tables,error,reference.answer,feedback.correctness_evaluator,feedback.groundedness_evaluator,feedback.retrieval_relevance_evaluator,execution_time,example_id,id
0,Recommend the cheapest unlimited global roamin...,"Based on the retrieved plans, the plan with un...",[page_content='Source Table: plans\nPlan Id: 5...,[plans],None,If no unlimited global roaming plan is present...,True,True,False,4.456945,07ccefe6-f4bf-4e52-b930-25cf8a0ee7b9,019e4694-6cc6-7bc2-a9b9-f71a8e8e346a
1,Recommend an offer for a VIP customer with roa...,"Based on the retrieved plans data, for a VIP c...",[page_content='Source Table: plans\nPlan Id: 5...,"[plans, support_interactions]",None,Retrieve roaming add-ons and high-value or VIP...,True,True,True,6.592045,0ffccb77-73b2-463d-9e08-1e0e1956354b,019e4694-6cc7-7d30-8d85-638c83c9be6e
2,Recommend a campaign for a high-value customer.,To recommend a campaign for a high-value custo...,[page_content='Source Table: campaigns\nCampai...,[campaigns],None,"Retrieve campaigns targeting high-value, VIP, ...",False,True,False,4.321432,1b455b0c-74c5-4ce0-b515-004baefc5f86,019e4694-8688-73b2-8d20-7755c8995ffe
3,Recommend a customer-care message for a billin...,Based on the reviewed complaints data related ...,[page_content='Source Table: complaints\nCompl...,[complaints],None,"Draft a short, professional, customer-friendly...",False,True,True,5.186995,2d60aee4-ecef-49ff-b2ad-810455650419,019e4694-981f-7dc3-9dfc-e61024853957
4,Which add-on is suitable for a customer who tr...,"Based on the retrieved addons data, here is an...",[page_content='Source Table: addons\nAddon Id:...,[addons],None,Recommend a roaming-related add-on if availabl...,True,True,True,6.639218,195d9cd8-c31e-45cb-9147-477dc2caacb2,019e4694-7e2f-77e3-bd36-dd258798c90b
5,Suggest a retention action for a customer with...,Based on the retrieved customer satisfaction d...,[page_content='Source Table: customer_satisfac...,[customer_satisfaction],None,"Use satisfaction, support, and campaign contex...",True,True,False,7.195484,279cbbc8-c43e-438f-b5f0-441829c3fb00,019e4694-976a-7081-a851-4d965e665223
6,Which is the best plan for heavy data users wh...,"For heavy data users who are likely to churn, ...",[page_content='Source Table: plans\nPlan Id: 5...,[plans],None,Recommend a high-data plan or relevant retenti...,True,True,True,5.363869,375ff1ea-0656-4c13-adef-a09922001ead,019e4694-ac62-73f3-b9db-ae6d85d569b2
7,Summarize all plan prices and rank them cheape...,"Based on the retrieved plans, here is the summ...",[page_content='Source Table: plans\nPlan Id: 5...,[plans],None,The answer should identify this as a structure...,False,True,True,4.741066,46507329-630d-421c-9b59-edce05aef53c,019e4694-b385-77e2-81e8-8c8a085b2363
8,Recommend a plan for a heavy mobile data user.,"For a heavy mobile data user, I recommend the ...",[page_content='Source Table: plans\nPlan Id: 9...,[plans],None,Recommend a high-data mobile plan. The answer ...,True,True,True,3.934217,55cc8e43-fe10-4ad0-ad6e-18d4f43b22d9,019e4694-c157-7902-8c11-9d1e20101a36
9,Recommend a plan for a low-cost customer.,For a low-cost customer looking for an afforda...,[page_content='Source Table: plans\nPlan Id: 1...,[plans],None,Recommend a lower-cost or value-oriented plan ...,True,True,True,3.160677,5e2a2b67-f85f-4c71-9437-9a11274be2d0,019e4694-c60b-74f2-a621-ecd5621dbecd


In [ ]:
# ============================================================
# 14. Optional: Compare model experiments
# ============================================================
# To compare models:
# 1. Change RAG_MODEL_NAME to "gpt-4.1" above
# 2. Re-run cells 7, 10, 12, and 13
# 3. Compare experiments in LangSmith UI
#
# Teaching message:
# LangSmith lets you compare prompts, models, retriever settings,
# and document quality systematically.


# Teaching Script

Use this explanation in class:

> In Class 4, we built a RAG assistant.  
> Now we ask: how do we know if the RAG assistant is actually good?

A single good answer is not enough. We need a small test set.

That test set is called a **golden dataset**.

For each question, we store:

- the question
- expected answer criteria
- expected source tables
- difficulty level
- skill being tested

Then LangSmith runs our RAG assistant on the dataset and checks:

1. **Correctness** — does the answer match the expected criteria?
2. **Groundedness** — is the answer supported by retrieved documents?
3. **Retrieval relevance** — did the retriever bring useful documents?

Best teaching line:

> RAG is not complete when it gives an answer.  
> RAG is complete when we can evaluate whether the answer is correct, grounded, and useful.
